<a href="https://colab.research.google.com/github/Ahmed-Hamam/miniimagenet-small-scratch-results/blob/main/CUB-Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CUB-200-2011 — Plain ProtoNet (MobileNetV3-SMALL, FROM SCRATCH) — KAGGLE VERSION
# ============================================================================
# نسخة CUB من سكريبت miniImageNet-Small-scratch: الـ backbone MobileNetV3-Small
# بيتدرّب **من الصفر** (weights=None، من غير ImageNet pretraining) على الـ 100
# صنف train بس، عند 84 و224 — للرد على اعتراض إن ImageNet pretraining بيدخّل
# معلومات عن أصناف الطيور (ImageNet فيه أصناف طيور كتير متداخلة مع CUB).
# لو مكسب الدقة من الدقة الأعلى فضل موجود → الاعتراض اتردّ عليه بالدليل.
#
# كل حاجة تانية زي نسخة CUB-Large: Kaggle input، progress.json، push/restore من
# GitHub، فحص اكتمال الـ split، matched-episode paired test.
#
# ⚠️ ملاحظات:
#  1) التدريب من الصفر على ~6000 صورة بس: الدقة هتبقى أقل بكتير من pretrained،
#     وده متوقع. STAGE1_EPOCHS=80 افتراضيًا (عدّله تحت).
#  2) الـ checkpoints الوسيطة لكل epoch في /kaggle/working/_resume_tmp (برّه
#     WORK_ROOT → مش بتتعمل push)، وبتكمّل لو الجلسة اتقطعت.
#  3) لازم تغيّر GITHUB_REPO لمستودعك.
# ============================================================================

# ============================================================================
# 🆕 الفروق عن نسخة Colab (باقي الكود **زي ما هو 100%**):
#    1) مفيش kagglehub خالص — الداتاست بتاعك متوقع يكون مرفوع كـ Kaggle
#       Dataset ومربوط كـ Input للـ notebook (موجود تلقائيًا في
#       /kaggle/input/...)، فالكود بيدوّر عليه هناك مباشرة بدل ما ينزّله.
#    2) مفيش Google Drive على Kaggle، فالاستمرارية بين الجلسات بتتم بطريقة
#       مختلفة: كل حاجة (progress.json + كل الـ checkpoints) بتتحفظ في
#       /kaggle/working (ده بيفضل موجود طول الـ session، وبيتحفظ تلقائيًا
#       لما تعمل "Save Version"). لو عايز تكمّل في جلسة/notebook جديد بعد
#       كده، لازم تاخد الـ Output بتاع النسخة المحفوظة وتضيفه كـ **Input**
#       تاني للـ notebook الجديد — والكود هنا بيدوّر عليه تلقائيًا ويسترجع
#       منه قبل ما يبدأ.
#
#    شرح خطوة بخطوة إزاي تعمل كده (اقرأه قبل أول تشغيل):
#    ───────────────────────────────────────────────────
#    أ) أول مرة: شغّل النوتبوك عادي. كل التدريب هيحفظ نتائجه أولًا بأول في
#       /kaggle/working/cub200_workdir (progress.json + checkpoints).
#
#    ب) قبل ما الـ session تقرب تخلص (أو وقت ما حبيت)، اعمل:
#       "Save Version" (فوق يمين النوتبوك) → اختار "Save & Run All (Commit)"
#       → استنى لحد ما يخلص. ده بيحفظ محتوى /kaggle/working كـ **Output**
#       بتاع النسخة دي.
#
#    ج) في الجلسة/النوتبوك الجديد:
#       - روح لصفحة النوتبوك بتاعك → تبويب "Output" → هتلاقي فيها نفس
#         ملفات /kaggle/working من آخر نسخة محفوظة (progress.json +
#         checkpoints).
#       - من شاشة "Add Input" (يمين الشاشة) دوّر على اسم النوتبوك بتاعك
#         نفسه (مش الداتاست) وضيفه كـ Input. Kaggle هيضيفه في
#         /kaggle/input/<اسم-النوتبوك>/ وهيكون فيه كل ملفات الـ Output
#         اللي اتحفظت.
#       - شغّل الكود عادي — هو هيدوّر تلقائيًا في /kaggle/input على أي
#         مجلد فيه progress.json، ولو لقاه هينسخه محليًا لـ
#         /kaggle/working ويكمّل بالظبط من عنده (نفس منطق الـ Resume
#         الأصلي، من غير أي تدريب أو تقييم مكرر).
#
#    ⚠️ ملحوظة: لازم تعمل الخطوة (ب) بانتظام (مش بس في الآخر)، لأن لو الـ
#       session اتقطعت فجأة من غير ما تعمل Save Version، هتفقد أي تقدّم
#       من غير commit. يفضل تعمل commit كل ما تخلص seed أو input_size.
#
# 🆕 AUTO-PUSH TO GITHUB (بديل تلقائي لخطوة ب/ج اليدوية):
#    بعد ما كل seed يخلص، الكود بيعمل push تلقائي لملفات progress.json +
#    الـ checkpoints (backbone + best_*.pth) بتاعة الـ seed ده لمستودع
#    GitHub بتاعك — من غير أي تدخل يدوي. تقدر بعد كده تنزّل الملفات من
#    GitHub على جهازك وقت ما تحب (عادي، من صفحة المستودع، أو بـ git pull).
#
#    خطوات الإعداد (مرة واحدة بس):
#    ───────────────────────────────
#    1) اعمل مستودع (Repository) جديد وخاص (Private) على GitHub، مخصّص
#       للنتائج بس (عشان الملفات الكبيرة متتراكمش في مستودع الكود الأساسي).
#       مثلًا: github.com/<username>/cub200-protonet-results
#
#    2) اعمل Personal Access Token من GitHub:
#       Settings → Developer settings → Personal access tokens →
#       Tokens (classic) → Generate new token → اختار صلاحية "repo" بس.
#       انسخ الـ token (مش هيتعرض تاني بعد كده).
#
#    3) في Kaggle، من "Add-ons" → "Secrets" في النوتبوك:
#       ضيف Secret جديد باسم GITHUB_TOKEN وقيمته الـ token اللي نسخته.
#
#    4) حدّد اسم المستودع بتاعك في المتغير GITHUB_REPO تحت (شكل
#       "username/reponame")، وتأكد إن "Internet" مفعّل في إعدادات
#       النوتبوك (الموجودة في الشريط الجانبي تحت "Session options").
#
#    5) شغّل الكود عادي — بعد كل seed، هيتعمل push تلقائي. تقدر تتابع
#       النتائج من صفحة المستودع على GitHub في أي وقت.
#
#    ⚠️ ملحوظات:
#    - لو الـ push فشل لأي سبب (مفيش إنترنت، الـ secret غير موجود، ...)،
#      الكود بيطبع تحذير ويكمّل التدريب عادي من غير ما يوقف — الملفات
#      برضه موجودة في /kaggle/working وهتتحفظ مع Save Version زي الأول.
#    - الـ checkpoints حجمها كبير نسبيًا (عشرات الـ MB)، فمع تكرار الـ
#      push بعد كل seed، حجم المستودع على GitHub هيكبر تدريجيًا — ده
#      طبيعي ومتوقع لمستودع نتائج مخصّص، لكن يفضل تفضل متابعه.
# ============================================================================

import os
import json
import glob
import shutil
import subprocess

try:
    from thop import profile as thop_profile
except ImportError:
    print("📦 Installing thop...")
    os.system('pip install -q thop')
    from thop import profile as thop_profile

try:
    from scipy import stats as scipy_stats
except ImportError:
    print("📦 Installing scipy...")
    os.system('pip install -q scipy')
    from scipy import stats as scipy_stats

import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image

# ============================================
# 0. PATHS (Kaggle: input للداتاست، working للتدريب/الحفظ)
# ============================================
# الداتاست: من /kaggle/input مباشرة (مفيش تنزيل خالص)
KAGGLE_INPUT_ROOT = "/kaggle/input"

# مكان العمل: /kaggle/working (بيتحفظ لما تعمل Save Version)
WORK_ROOT      = "/kaggle/working/cub200_small_scratch_workdir"
DATA_ROOT      = "/kaggle/working/cub200_data_local"
RESUME_TMP     = "/kaggle/working/_resume_tmp"   # برّه WORK_ROOT → مش بيتعمله push
os.makedirs(RESUME_TMP, exist_ok=True)

# ===== إعدادات التدريب من الصفر (عدّلها هنا) =====
STAGE1_EPOCHS   = 80
WARMUP_EPOCHS   = 3
BASE_LR_PER_256 = 0.1      # lr = BASE_LR_PER_256 * batch/256
STAGE1_WD       = 5e-4
USE_AMP         = True
CUB_SPLIT_ROOT = os.path.join(DATA_ROOT, "cub200_split")
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print(f"📂 KAGGLE_INPUT_ROOT (الداتاست + أي نسخة سابقة) = {KAGGLE_INPUT_ROOT}")
print(f"📂 WORK_ROOT (progress + checkpoints, بيتحفظ مع Save Version) = {WORK_ROOT}")

# 🆕 إعداد GitHub Auto-Push — غيّر القيمة دي لاسم مستودعك (username/reponame)
GITHUB_REPO       = "Ahmed-Hamam/miniimagenet-small-scratch-results"  # ⚠️ غيّرها!
GITHUB_BRANCH     = "main"
GITHUB_CLONE_DIR  = "/kaggle/working/_github_results_repo"
GITHUB_SUBFOLDER  = "cub200_small_scratch_results"  # المجلد جوه المستودع اللي هتتحط فيه الملفات


def _run_git_cmd(args, cwd=None, env=None):
    """ينفّذ أمر git ويرجّع (success, output). بيلتقط stderr/stdout في
    الرسالة نفسها عشان نقدر نطبع سبب الفشل لو حصل. (معرّفة هنا فوق عشان
    تُستخدم في الاسترجاع التلقائي من GitHub بالأول، وكمان في push_results_to_github تحت)."""
    result = subprocess.run(
        ["git"] + args, cwd=cwd, env=env,
        capture_output=True, text=True
    )
    ok = (result.returncode == 0)
    output = (result.stdout or "") + (result.stderr or "")
    return ok, output


def _restore_previous_workdir_from_input():
    """🆕 يدوّر في /kaggle/input على أي مجلد مضاف فيه progress.json (يعني
    ده Output بتاع نسخة سابقة من نفس النوتبوك أُضيف كـ Input)، ولو لقاه
    بينسخ كل محتوياته لـ WORK_ROOT محليًا (نسخ محلي سريع، مش عبر الشبكة)
    عشان الـ Resume يشتغل عادي."""
    if os.path.exists(os.path.join(WORK_ROOT, "progress.json")):
        print("✅ progress.json موجود بالفعل في WORK_ROOT — مفيش حاجة نسترجعها.")
        return

    candidates = glob.glob(os.path.join(KAGGLE_INPUT_ROOT, "*", "**", "progress.json"),
                            recursive=True)
    if not candidates:
        print("ℹ️  مفيش نسخة سابقة (progress.json) موجودة في أي Input — هنبدأ من الصفر.")
        return

    # لو فيه أكتر من واحد (مش متوقع)، ناخد أحدث واحد بالـ mtime
    src_progress = max(candidates, key=os.path.getmtime)
    src_dir = os.path.dirname(src_progress)
    print(f"♻️  لقينا نسخة سابقة في: {src_dir}")
    print("📦 بننسخ progress.json وكل الـ checkpoints لـ WORK_ROOT محليًا...")

    for item in os.listdir(src_dir):
        src_path = os.path.join(src_dir, item)
        dst_path = os.path.join(WORK_ROOT, item)
        if os.path.isfile(src_path) and not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)

    print(f"✅ تم استرجاع {len(os.listdir(WORK_ROOT))} ملف من النسخة السابقة.")


def _restore_previous_workdir_from_github():
    """🆕 لو مفيش نسخة سابقة في /kaggle/input (يعني أول تشغيل في notebook
    جديد، أو نسيت تضيف الـ Input يدويًا)، نحاول نسحب آخر progress.json +
    checkpoints مباشرة من مستودع GitHub (لو GITHUB_REPO متظبط ومتاح الـ
    GITHUB_TOKEN). ده بيخلي الاستمرارية تلقائية بالكامل، من غير ما تحتاج
    تضيف Input يدوي كل مرة."""
    if os.path.exists(os.path.join(WORK_ROOT, "progress.json")):
        return  # اتسترجعت بالفعل من /kaggle/input

    if GITHUB_REPO.startswith("your-username/"):
        print("ℹ️  GITHUB_REPO لسه مش معدّل — هنتجاهل الاسترجاع التلقائي من GitHub.")
        return

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print(f"ℹ️  مقدرتش أجيب GITHUB_TOKEN ({e}) — هنتجاهل الاسترجاع التلقائي "
              f"من GitHub ونبدأ من الصفر (أو من /kaggle/input لو موجود).")
        return

    print(f"\n🔍 بنحاول نسترجع نسخة سابقة من مستودع GitHub ({GITHUB_REPO})...")
    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"

    try:
        if not os.path.exists(os.path.join(GITHUB_CLONE_DIR, ".git")):
            ok, out = _run_git_cmd([
                "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                remote_url, GITHUB_CLONE_DIR
            ])
            if not ok:
                print(f"ℹ️  مقدرناش نعمل clone للمستودع (غالبًا فاضي أو أول مرة) "
                      f"— هنبدأ من الصفر:\n{out}")
                return
        else:
            ok, out = _run_git_cmd(["pull", "--rebase", "origin", GITHUB_BRANCH],
                                    cwd=GITHUB_CLONE_DIR)

        src_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        src_progress = os.path.join(src_dir, "progress.json")
        if not os.path.exists(src_progress):
            print("ℹ️  مفيش progress.json جوه المستودع — هنبدأ من الصفر.")
            return

        print(f"♻️  لقينا نسخة سابقة على GitHub في {src_dir}")
        print("📦 بننسخ progress.json وكل الـ checkpoints لـ WORK_ROOT محليًا...")
        for item in os.listdir(src_dir):
            src_path = os.path.join(src_dir, item)
            dst_path = os.path.join(WORK_ROOT, item)
            if os.path.isfile(src_path) and not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)

        print(f"✅ تم استرجاع {len(os.listdir(WORK_ROOT))} ملف من GitHub — "
              f"الكود هيكمّل من عند آخر seed اتسجّل بدل ما يبدأ من الصفر.")

    except Exception as e:
        print(f"⚠️  الاسترجاع التلقائي من GitHub فشل بشكل غير متوقع ({e}) "
              f"— هنبدأ من الصفر.")


_restore_previous_workdir_from_input()
_restore_previous_workdir_from_github()

# ============================================
# 1. SETUP
# ============================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Processing on: {device}")

STAGE_TIMES = {}

class StageTimer:
    """Context manager that records wall-clock time for a named stage."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        print(f"\n⏱️  [{self.name}] started...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self.t0
        STAGE_TIMES[self.name] = elapsed
        mins = elapsed / 60.0
        print(f"⏱️  [{self.name}] finished in {elapsed:.1f}s ({mins:.2f} min)")


def print_stage_time_summary():
    if not STAGE_TIMES:
        print("No stage timings recorded yet.")
        return
    print("\n" + "="*40)
    print("⏱️  STAGE TIME SUMMARY")
    print("="*40)
    total = 0.0
    for name, secs in STAGE_TIMES.items():
        print(f"  {name:45s} {secs:8.1f}s  ({secs/60:.2f} min)")
        total += secs
    print(f"  {'TOTAL':45s} {total:8.1f}s  ({total/60:.2f} min)")

# ============================================
# 2. MODEL PROFILING: PARAMETERS & FLOPs
# ============================================

def count_parameters(model, trainable_only=False):
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def compute_flops(backbone_module, input_size, device=device):
    """Backbone-only FLOPs/params via thop (feature extraction pass only)."""
    backbone_module = backbone_module.to(device).eval()
    dummy = torch.randn(1, 3, input_size, input_size).to(device)

    class _FeatureOnly(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone

        def forward(self, x):
            x = self.backbone.features(x)
            x = self.backbone.avgpool(x)
            return torch.flatten(x, 1)

    wrapper = _FeatureOnly(backbone_module).to(device).eval()
    with torch.no_grad():
        macs, params = thop_profile(wrapper, inputs=(dummy,), verbose=False)

    gflops = macs / 1e9
    params_m = params / 1e6
    print(f"📐 Backbone profile @ {input_size}x{input_size}: "
          f"{params_m:.2f}M params | {gflops:.4f} GFLOPs (MACs)")
    return {"params": params, "params_M": params_m, "macs": macs, "gflops": gflops}


def report_full_model_stats(model, input_size):
    """Total params of ProtoNet wrapper (backbone only, no extra params) + FLOPs."""
    total_params = count_parameters(model)
    trainable_params = count_parameters(model, trainable_only=True)
    print(f"\n📊 ProtoNet full model: "
          f"{total_params/1e6:.3f}M total params | "
          f"{trainable_params/1e6:.3f}M trainable params")
    flops_info = compute_flops(model.backbone, input_size=input_size)
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        **flops_info
    }

# ============================================
# 3. DATA PREPARATION — CUB-200-2011 (من /kaggle/input مباشرة)
# ============================================

def _find_images_folder(root):
    """أبسط وأوثق طريقة: يدوّر تحديدًا على مجلد اسمه 'images' (زي بنية
    CUB_200_2011/images/ الرسمية) وفيه مجلدات فرعية (الأصناف)."""
    for r, dirs, files in os.walk(root):
        if os.path.basename(r) == "images" and any(
            os.path.isdir(os.path.join(r, d)) for d in dirs
        ):
            return r
    return None


def _find_class_dirs(root):
    """Return a directory whose immediate children are all class-folders
    (heuristic: at least 50 subdirs, each containing images)."""
    for r, dirs, files in os.walk(root):
        if not dirs:
            continue
        sub_dirs = [d for d in dirs if os.path.isdir(os.path.join(r, d))]
        if len(sub_dirs) < 50:  # CUB has 200 classes total / per split >=50
            continue
        sample = os.path.join(r, sub_dirs[0])
        has_imgs = any(f.lower().endswith(('jpg', 'jpeg', 'png'))
                       for f in os.listdir(sample))
        if has_imgs:
            return r
    return None


def _find_cub_dataset_root():
    """يدوّر في /kaggle/input على مجلد الداتاست بتاع CUB-200-2011 (اللي
    انت ضايفه كـ Input Dataset)، متجاهلًا أي مجلد فيه progress.json
    (ده مجلد نسخة سابقة من الـ Output، مش الداتاست نفسه)."""
    if not os.path.isdir(KAGGLE_INPUT_ROOT):
        return None
    for entry in sorted(os.listdir(KAGGLE_INPUT_ROOT)):
        entry_path = os.path.join(KAGGLE_INPUT_ROOT, entry)
        if not os.path.isdir(entry_path):
            continue
        if os.path.exists(os.path.join(entry_path, "progress.json")):
            continue  # ده مجلد نسخة سابقة (Output)، مش الداتاست
        if _find_images_folder(entry_path) or _find_class_dirs(entry_path):
            return entry_path
    return None


def _split_is_complete(split_root):
    """🆕 يتأكد إن الثلاثة فولدرات train/val/test موجودين فعلًا وفيهم
    أصناف (مش مجرد إن split_root نفسه موجود). بدون الفحص ده، لو حصل أي
    قطع أو مشكلة ديسك في نص بناء الداتاست (مثلًا train وval خلصوا وtest
    اتقطع)، الكود كان بيفتكر إن الداتاست 'جاهز' غلط ويكمّل بمجلد ناقص،
    وده كان سبب FileNotFoundError على مجلد test وقت الـ Matched-Episode
    Evaluation."""
    for split_name in ("train", "val", "test"):
        split_dir = os.path.join(split_root, split_name)
        if not os.path.isdir(split_dir):
            return False
        class_dirs = [d for d in os.listdir(split_dir)
                      if os.path.isdir(os.path.join(split_dir, d))]
        if not class_dirs:
            return False
    return True


def prepare_cub200():
    if os.path.exists(CUB_SPLIT_ROOT):
        if _split_is_complete(CUB_SPLIT_ROOT):
            print("✅ CUB-200-2011 split already present and looks complete "
                  "(train/val/test كلهم موجودين وفيهم أصناف).")
            return
        print("⚠️  لقينا مجلد split موجود لكنه ناقص (train/val/test مش كلهم "
              "كاملين — غالبًا بسبب انقطاع سابق في منتصف البناء). "
              "هنمسحه ونعيد بناء الداتاست بالكامل من /kaggle/input من جديد.")
        shutil.rmtree(CUB_SPLIT_ROOT)

    print("\n🔍 بندوّر على داتاست CUB-200-2011 في /kaggle/input...")
    dataset_path = _find_cub_dataset_root()
    if dataset_path is None:
        raise RuntimeError(
            "❌ مش لاقي داتاست CUB-200-2011 في /kaggle/input. اتأكد إنك "
            "ضفته كـ Input Dataset للنوتبوك ده (من شاشة Add Input)."
        )
    print(f"📁 Dataset found at: {dataset_path}")

    try:
        os.makedirs(CUB_SPLIT_ROOT, exist_ok=True)

        # الحالة 1: الداتاست موزّع بالفعل بمجلدات train/val/test رسمية
        existing_splits = {}
        for split_name in ["train", "val", "test"]:
            for r, dirs, files in os.walk(dataset_path):
                if os.path.basename(r).lower() == split_name and dirs:
                    existing_splits[split_name] = r
                    break

        if len(existing_splits) == 3:
            print("📦 Found official train/val/test split folders.")
            for split_name, src_root in existing_splits.items():
                dst_root = os.path.join(CUB_SPLIT_ROOT, split_name)
                if not os.path.exists(dst_root):
                    shutil.copytree(src_root, dst_root)
            print("✅ CUB-200-2011 Data Structure Ready! (official split found)")
            return

        # الحالة 2: مجلد واحد فيه كل الأصناف (200 مجلد، زي CUB_200_2011/images/)
        # → نعمل تقسيم يدوي class-disjoint: 100/50/50
        images_root = _find_images_folder(dataset_path) or _find_class_dirs(dataset_path)
        if images_root is None:
            raise RuntimeError(
                "Could not locate CUB-200-2011 class directories in dataset."
            )

        print(f"📁 Found images root: {images_root}")
        classes = sorted([d for d in os.listdir(images_root)
                           if os.path.isdir(os.path.join(images_root, d))])
        print(f"📦 Found {len(classes)} classes.")

        rng = random.Random(42)  # تقسيم ثابت وقابل لإعادة الإنتاج بين الجلسات —
                                  # غيّر القيمة لو عايز تقسيم مختلف.
        rng.shuffle(classes)
        n_total = len(classes)
        # نسبة قياسية لأبحاث Few-Shot على CUB: 100 train / 50 val / 50 test
        # (لو عدد الأصناف الفعلي مختلف عن 200 بسبب اختلاف نسخة الداتاست، بنقسم
        # بنفس النسبة تقريبًا: 50% / 25% / 25%)
        if n_total == 200:
            n_train, n_val = 100, 50
        else:
            n_train = int(round(n_total * 0.5))
            n_val   = int(round(n_total * 0.25))
        splits = {
            "train": classes[:n_train],
            "val":   classes[n_train:n_train + n_val],
            "test":  classes[n_train + n_val:],
        }
        print(f"   Split sizes → train: {len(splits['train'])}, "
              f"val: {len(splits['val'])}, test: {len(splits['test'])}")

        for split_name, split_classes in splits.items():
            os.makedirs(os.path.join(CUB_SPLIT_ROOT, split_name), exist_ok=True)
            for c in split_classes:
                src = os.path.join(images_root, c)
                dst = os.path.join(CUB_SPLIT_ROOT, split_name, c)
                if not os.path.exists(dst):
                    shutil.copytree(src, dst)
        print("✅ CUB-200-2011 Data Structure Ready! (class-disjoint split)")
    except Exception as e:
        print(f"❌ Error preparing CUB-200-2011: {e}")
        raise


with StageTimer("Stage 0: Data Preparation (CUB-200-2011)"):
    prepare_cub200()

def _dataset_sanity_info():
    try:
        n_train = len(os.listdir(os.path.join(CUB_SPLIT_ROOT, "train")))
        print(f"📋 train classes = {n_train} (المتوقع 100 في تقسيم 100/50/50)")
        sizes = []
        for root, _, files in os.walk(os.path.join(CUB_SPLIT_ROOT, "train")):
            for fn in files[:1]:
                try:
                    sizes.append(Image.open(os.path.join(root, fn)).size)
                except Exception:
                    pass
            if len(sizes) >= 20:
                break
        if sizes:
            print(f"📋 عينة أحجام الصور الأصلية: {sizes[:3]} ... "
                  f"(أقصر ضلع max={max(min(w, h) for w, h in sizes)})")
            if max(max(w, h) for w, h in sizes) <= 100:
                print("⚠️  الصور الأصلية صغيرة (≤100px) → 224 هو upsampling فقط.")
    except Exception as e:
        print(f"(sanity check skipped: {e})")

_dataset_sanity_info()

# ============================================
# 4. PROTOTYPE (Plain ProtoNet — no learnable norm, no gamma)
# ============================================

def compute_prototype(z_support):
    """Class prototype = arithmetic mean of raw support embeddings."""
    return z_support.mean(dim=0)

# ============================================
# 5. MODEL DEFINITION — MobileNetV3-SMALL (from scratch)
# ============================================

def get_mobilenet_finetuned(num_classes=100, supervised=True):
    # FROM SCRATCH: weights=None → مفيش أي ImageNet pretraining
    print(f"🔄 Building MobileNetV3-SMALL FROM SCRATCH (num_classes={num_classes})...")
    model = models.mobilenet_v3_small(weights=None)

    if supervised:
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes)
    else:
        model.classifier = nn.Identity()

    return model


def build_protonet_for_inference(checkpoint_path, num_classes=100):
    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.classifier = nn.Identity()
    model = MobileProtoNet(base_model)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✅ Loaded checkpoint: {checkpoint_path}")
    return model


class MobileProtoNet(nn.Module):
    """
    Plain ProtoNet: prototype = mean of support embeddings,
    logits = -squared_euclidean_distance(query, prototype).
    لا يوجد LearnableNorm ولا gamma (بدون أي معاملات إضافية قابلة للتعلم
    خارج الـ backbone نفسه).
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, support_x, query_x, n_way, k_shot):
        x = torch.cat([support_x, query_x], dim=0)
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        z = torch.flatten(x, 1)

        z_support = z[:support_x.size(0)]
        z_query   = z[support_x.size(0):]

        z_proto = torch.stack([
            compute_prototype(z_support[i * k_shot:(i + 1) * k_shot])
            for i in range(n_way)
        ])

        # ProtoNet القياسي يستخدم مربع المسافة الإقليدية
        dists = torch.cdist(z_query, z_proto) ** 2
        logits = -dists
        return logits

# ============================================
# 6. DATA LOADERS (parametrized by input_size)
# ============================================

def get_resize_dim(crop_dim):
    return round(crop_dim * 256 / 224)


def build_transforms(input_size):
    resize_dim = get_resize_dim(input_size)

    train_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.RandomCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, test_transform


CUB_ROOT = CUB_SPLIT_ROOT

# batch size أقل عند الدقات الكبيرة عشان نتجنب overflow في VRAM. القيم دي
# Small خفيف، فنقدر نكبّر الـ batch (مع lr متناسب في Stage 1).
# في الذاكرة (~5.4M params وFLOPs أعلى بشكل ملحوظ من Small). القيم تقديرية
# — لو حصل CUDA Out Of Memory قلّلها أكتر حسب الـ GPU المتاح فعليًا.
BATCH_SIZE_BY_RESOLUTION = {84: 128, 112: 96, 168: 64, 224: 64, 256: 48}


class EpisodicSampler:
    def __init__(self, root, transform):
        self.root      = root
        self.transform = transform
        self.classes   = sorted(os.listdir(root))
        self.data      = {}
        for c in self.classes:
            cls_path = os.path.join(root, c)
            self.data[c] = [
                os.path.join(cls_path, img)
                for img in os.listdir(cls_path)
                if img.lower().endswith(('jpg', 'jpeg', 'png'))
            ]

    def get_episode(self, n_way, k_shot, q_query):
        selected_classes = random.sample(self.classes, n_way)
        support_imgs, query_imgs, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = self.data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = random.choices(available_imgs, k=needed)
            else:
                sampled_paths = random.sample(available_imgs, needed)

            for path in sampled_paths[:k_shot]:
                img = Image.open(path).convert('RGB')
                support_imgs.append(self.transform(img))

            for path in sampled_paths[k_shot:]:
                img = Image.open(path).convert('RGB')
                query_imgs.append(self.transform(img))
                query_labels.append(i)

        return (torch.stack(support_imgs),
                torch.stack(query_imgs),
                torch.tensor(query_labels))

# ============================================
# 6b. MATCHED-EPISODE EVALUATION (fixed episode set)
# ============================================

def generate_fixed_episodes(root, n_way, k_shot, q_query, n_episodes, seed):
    rng = random.Random(seed)  # RNG محلي مستقل — لا يمس حالة random العامة
    classes = sorted(os.listdir(root))
    data = {}
    for c in classes:
        cls_path = os.path.join(root, c)
        data[c] = [
            os.path.join(cls_path, img)
            for img in os.listdir(cls_path)
            if img.lower().endswith(('jpg', 'jpeg', 'png'))
        ]

    episodes = []
    for _ in range(n_episodes):
        selected_classes = rng.sample(classes, n_way)
        support_paths, query_paths, query_labels = [], [], []

        for i, cls in enumerate(selected_classes):
            available_imgs = data[cls]
            needed = k_shot + q_query
            if len(available_imgs) < needed:
                sampled_paths = rng.choices(available_imgs, k=needed)
            else:
                sampled_paths = rng.sample(available_imgs, needed)

            support_paths.extend(sampled_paths[:k_shot])
            query_paths.extend(sampled_paths[k_shot:])
            query_labels.extend([i] * q_query)

        episodes.append({
            "classes": selected_classes,
            "support_paths": support_paths,
            "query_paths": query_paths,
            "query_labels": query_labels,
        })

    return episodes


def _load_episode_tensors(episode_def, transform, n_way, k_shot):
    support_imgs = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["support_paths"]]
    query_imgs   = [transform(Image.open(p).convert('RGB'))
                     for p in episode_def["query_paths"]]
    query_labels = torch.tensor(episode_def["query_labels"])
    return torch.stack(support_imgs), torch.stack(query_imgs), query_labels


@torch.no_grad()
def evaluate_fixed_episodes(model, episode_defs, transform, n_way, k_shot, desc=""):
    """يقيّم موديل على مجموعة episodes ثابتة (paths محددة سلفًا) ويرجّع
    مصفوفة دقة لكل episode على حدة (بدون تجميع)، عشان تُستخدم في اختبار
    paired لاحقًا."""
    model.eval()
    accs = []
    for ep_def in tqdm(episode_defs, desc=f"Matched-Eval {desc}", leave=False):
        sx, qx, qy = _load_episode_tensors(ep_def, transform, n_way, k_shot)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)
        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())
    return np.array(accs)


def paired_resolution_test(acc_84, acc_224, label=""):
    """اختبار paired بين نفس الـ episodes بدقتين مختلفتين.
    Δ_i = Acc224_i - Acc84_i ، H0: E[Δ] = 0."""
    assert len(acc_84) == len(acc_224), "عدد الـ episodes لازم يتطابق للمقارنة الـ paired"
    delta = acc_224 - acc_84
    mean_delta = delta.mean()
    std_delta  = delta.std(ddof=1)
    n = len(delta)
    se = std_delta / np.sqrt(n)
    ci95 = 1.96 * se

    t_stat, t_pval = scipy_stats.ttest_rel(acc_224, acc_84)
    try:
        w_stat, w_pval = scipy_stats.wilcoxon(acc_224, acc_84)
    except ValueError:
        # Wilcoxon بيفشل لو كل الفروق = صفر
        w_stat, w_pval = float('nan'), float('nan')

    print(f"\n📈 Paired 84x84 vs 224x224 — {label}")
    print(f"   n episodes        : {n}")
    print(f"   Mean Acc @84       : {acc_84.mean():.2%}")
    print(f"   Mean Acc @224      : {acc_224.mean():.2%}")
    print(f"   Mean Δ (224-84)    : {mean_delta:+.2%}  (95% CI ±{ci95:.2%})")
    print(f"   Paired t-test      : t={t_stat:.4f}, p={t_pval:.6f}")
    print(f"   Wilcoxon signed-rank: W={w_stat:.4f}, p={w_pval:.6f}")

    return {
        "n": n,
        "mean_delta": mean_delta,
        "ci95_delta": ci95,
        "t_stat": t_stat,
        "t_pval": t_pval,
        "w_stat": w_stat,
        "w_pval": w_pval,
        "acc_84_mean": acc_84.mean(),
        "acc_224_mean": acc_224.mean(),
    }

# ============================================
# 7. EXECUTION STAGES
# ============================================

def stage1_finetuning(train_loader_simple, epochs=STAGE1_EPOCHS, num_classes=100, tag="run"):
    print(f"\n🦾 STAGE 1: Training MobileNetV3-Small FROM SCRATCH on the 100 train classes "
          f"for {epochs} epochs...")

    model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True).to(device)
    bs    = train_loader_simple.batch_size
    lr    = BASE_LR_PER_256 * bs / 256
    print(f"   lr={lr:.4f} (batch={bs}) | warmup={WARMUP_EPOCHS} ep | wd={STAGE1_WD} | AMP={USE_AMP}")
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                                 nesterov=True, weight_decay=STAGE1_WD)
    criterion = nn.CrossEntropyLoss()
    use_amp   = USE_AMP and str(device).startswith("cuda")
    scaler    = torch.amp.GradScaler("cuda", enabled=use_amp)

    import math
    def lr_at(epoch):
        if epoch < WARMUP_EPOCHS:
            return lr * (epoch + 1) / WARMUP_EPOCHS
        t = (epoch - WARMUP_EPOCHS) / max(1, epochs - WARMUP_EPOCHS)
        return 0.5 * lr * (1 + math.cos(math.pi * t))

    resume_path = os.path.join(RESUME_TMP, f"stage1_resume_{tag}.pth")
    start_epoch = 0
    if os.path.exists(resume_path):
        try:
            ck = torch.load(resume_path, map_location=device)
            model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["opt"])
            start_epoch = ck["epoch"]
            print(f"♻️  Resuming Stage 1 from epoch {start_epoch}/{epochs}")
        except Exception as e:
            print(f"⚠️  resume checkpoint تالف ({e}) — هنبدأ من الأول.")

    for epoch in range(start_epoch, epochs):
        for g in optimizer.param_groups:
            g["lr"] = lr_at(epoch)
        model.train()
        total_loss, correct, total = 0, 0, 0

        pbar = tqdm(train_loader_simple, desc=f"Scratch Ep {epoch+1}/{epochs}", leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=("cuda" if str(device).startswith("cuda") else "cpu"), enabled=use_amp):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total   += labels.size(0)
            pbar.set_postfix({'Acc': f"{correct/total:.1%}"})

        print(f"   Ep {epoch+1}: LR {lr_at(epoch):.4f} | Loss {total_loss/len(train_loader_simple):.4f} | "
              f"Train Acc {correct/total:.2%}")
        tmp = resume_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": optimizer.state_dict(),
                    "epoch": epoch + 1}, tmp)
        os.replace(tmp, resume_path)

    print("✅ Scratch training complete.")
    return model


def stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                      n_way=5, k_shot=1, episodes=2000, num_classes=100,
                      ckpt_path="best_mobile_proto_cub200.pth"):
    print(f"\n🧠 STAGE 2: Meta-Training (ProtoNet) on CUB-200-2011 "
          f"{n_way}-way {k_shot}-shot...")

    base_model = get_mobilenet_finetuned(num_classes=num_classes, supervised=True)
    base_model.load_state_dict(pretrained_state_dict)
    base_model.classifier = nn.Identity()

    model = MobileProtoNet(base_model).to(device)

    # لا يوجد learnable_norm أو gamma الآن، لذلك مجموعة معاملات واحدة فقط للـ backbone
    optimizer = torch.optim.Adam(model.backbone.parameters(), lr=1e-5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience     = 0

    for ep in tqdm(range(episodes), desc="Meta-Train"):
        sx, qx, qy = train_sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        model.train()
        optimizer.zero_grad()
        logits = model(sx, qx, n_way, k_shot)
        loss   = criterion(logits, qy)
        loss.backward()
        optimizer.step()

        if (ep + 1) % 200 == 0:
            val_acc, _ = evaluate(model, val_sampler, n_way, k_shot,
                                   episodes=200, silent=True)
            print(f"\n   Ep {ep+1}: Loss {loss.item():.4f} | Val Acc: {val_acc:.2%}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)
                patience = 0
            else:
                patience += 1
                if patience > 10:
                    print("⏹️ Early stopping triggered.")
                    break

    return model


@torch.no_grad()
def evaluate(model, sampler, n_way, k_shot, episodes=500, silent=False):
    model.eval()
    accs     = []
    iterator = range(episodes) if silent else tqdm(range(episodes), desc="Evaluating")

    for _ in iterator:
        sx, qx, qy = sampler.get_episode(n_way, k_shot, q_query=15)
        sx, qx, qy = sx.to(device), qx.to(device), qy.to(device)

        logits = model(sx, qx, n_way, k_shot)
        preds  = logits.argmax(dim=1)
        accs.append((preds == qy).float().mean().item())

    mean_acc = np.mean(accs)
    ci95     = 1.96 * np.std(accs) / np.sqrt(episodes)

    if not silent:
        print(f"📊 Result: {mean_acc:.2%} ± {ci95:.2%}")

    return mean_acc, ci95

# ============================================
# 8. RUN ONE FULL EXPERIMENT (given input_size + seed)
# ============================================

NUM_TRAIN_CLASSES = 100  # CUB-200-2011 few-shot train split size (100/50/50)

def run_experiment(input_size, seed, epochs=STAGE1_EPOCHS, episodes=2000):
    print("\n" + "#"*60)
    print(f"### RUNNING EXPERIMENT: input_size={input_size}x{input_size} | seed={seed}")
    print("#"*60)

    set_seed(seed)

    train_transform, test_transform = build_transforms(input_size)

    train_dataset_simple = datasets.ImageFolder(
        root=os.path.join(CUB_ROOT, "train"),
        transform=train_transform
    )
    batch_size = BATCH_SIZE_BY_RESOLUTION.get(input_size, 32)
    train_loader_simple = DataLoader(
        train_dataset_simple,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )
    print(f"   Batch size: {batch_size}")

    train_sampler = EpisodicSampler(os.path.join(CUB_ROOT, "train"), test_transform)
    val_sampler   = EpisodicSampler(os.path.join(CUB_ROOT, "val"),   test_transform)
    test_sampler  = EpisodicSampler(os.path.join(CUB_ROOT, "test"),  test_transform)

    tag = f"{input_size}px_seed{seed}"

    # ── Stage 1 ──────────────────────────────
    backbone_ckpt = os.path.join(WORK_ROOT, f"backbone_scratch_cub200_small_{tag}.pth")
    if not os.path.exists(backbone_ckpt):
        with StageTimer(f"Stage 1: Fine-Tuning ({tag})"):
            ft_model = stage1_finetuning(train_loader_simple, epochs=epochs,
                                          num_classes=NUM_TRAIN_CLASSES, tag=tag)
            torch.save(ft_model.state_dict(), backbone_ckpt)
    else:
        print(f"✅ Pre-trained backbone found for {tag}. Skipping Stage 1.")
    pretrained_state_dict = torch.load(backbone_ckpt, map_location=device)

    results = {}

    for k_shot in [1, 5]:
        print("\n" + "="*40)
        print(f"🎯 TASK: 5-way {k_shot}-shot ({tag})")
        print("="*40)

        ckpt_path = os.path.join(
            WORK_ROOT, f"best_mobile_proto_cub200_small_scratch_{tag}_{k_shot}shot.pth"
        )

        with StageTimer(f"Stage 2: Meta-Training {k_shot}-shot ({tag})"):
            stage2_metatrain(pretrained_state_dict, train_sampler, val_sampler,
                              n_way=5, k_shot=k_shot, episodes=episodes,
                              num_classes=NUM_TRAIN_CLASSES, ckpt_path=ckpt_path)

        with StageTimer(f"Stage 3: Test Evaluation {k_shot}-shot ({tag})"):
            best_model = build_protonet_for_inference(
                checkpoint_path=ckpt_path, num_classes=NUM_TRAIN_CLASSES
            )
            mean_acc, ci95 = evaluate(best_model, test_sampler, n_way=5, k_shot=k_shot)
            results[k_shot] = {"mean_acc": mean_acc, "ci95": ci95, "ckpt_path": ckpt_path}

        if k_shot == 5:
            print("\n" + "="*40)
            print(f"📐 MODEL SIZE & COMPUTE REPORT ({tag})")
            print("="*40)
            report_full_model_stats(best_model, input_size=input_size)

    return results

# ============================================
# 8b. RESUME / PROGRESS TRACKING
# ============================================
# ملف تقدّم بيتحدّث بعد كل (input_size, seed, k_shot) يخلص — لو التجربة
# اتقطعت (انقطع النت، قفلت الجلسة، ...) وشغّلت الكود تاني (بعد ما تضيف
# الـ Output بتاع النسخة المحفوظة كـ Input زي ما اتشرح فوق)، هيقرأ الملف
# ده ويكمّل من عند آخر حاجة خلصها، من غير ما يعيد أي تدريب أو تقييم اتعمل
# قبل كده. الحفظ atomic (كتابة لملف مؤقت ثم rename) عشان لو الجلسة اتقفلت
# في نص الكتابة، الملف الأصلي يفضل سليم.

PROGRESS_FILE = os.path.join(WORK_ROOT, "progress.json")


def load_progress():
    """يرجّع (all_results, matched_accs) من ملف التقدّم لو موجود، وإلا
    قواميس فاضية. all_results بنفس شكل المتغيّر all_results في MAIN.
    matched_accs بنفس شكل matched_accs (مصفوفات per-episode لكل
    k_shot/seed/input_size، مستخدمة في الـ Matched-Episode Evaluation)."""
    if not os.path.exists(PROGRESS_FILE):
        return {}, {}
    try:
        with open(PROGRESS_FILE, "r") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"⚠️  ملف التقدّم {PROGRESS_FILE} تالف ({e}) — هنبدأ من الصفر.")
        return {}, {}

    all_results = {}
    for res_str, seeds in data.get("results", {}).items():
        res = int(res_str)
        all_results[res] = {}
        for seed_str, kshots in seeds.items():
            seed = int(seed_str)
            all_results[res][seed] = {}
            for k_str, v in kshots.items():
                all_results[res][seed][int(k_str)] = {
                    "mean_acc": v["mean_acc"],
                    "ci95": v["ci95"],
                    "ckpt_path": v["ckpt_path"],
                }

    matched_accs = {}
    for k_str, seeds in data.get("matched", {}).items():
        k_shot = int(k_str)
        matched_accs[k_shot] = {}
        for seed_str, res_dict in seeds.items():
            seed = int(seed_str)
            matched_accs[k_shot][seed] = {
                84:  np.array(res_dict["84"]),
                224: np.array(res_dict["224"]),
            }

    if all_results or matched_accs:
        print(f"\n♻️  لقينا تقدّم سابق في {PROGRESS_FILE}:")
        for res, seeds in all_results.items():
            print(f"    {res}px: seeds خلّصت = {sorted(seeds.keys())}")
        for k_shot, seeds in matched_accs.items():
            print(f"    Matched-Eval {k_shot}-shot: seeds خلّصت = {sorted(seeds.keys())}")

    return all_results, matched_accs


def save_progress(all_results, matched_accs):
    """يحفظ التقدّم الحالي بطريقة atomic (كتابة لملف مؤقت ثم rename)."""
    report = {
        "results": {
            str(res): {
                str(seed): {
                    str(k_shot): {
                        "mean_acc": float(v["mean_acc"]),
                        "ci95": float(v["ci95"]),
                        "ckpt_path": v["ckpt_path"],
                    }
                    for k_shot, v in kshots.items()
                }
                for seed, kshots in seeds.items()
            }
            for res, seeds in all_results.items()
        },
        "matched": {
            str(k_shot): {
                str(seed): {
                    "84":  res_dict[84].tolist(),
                    "224": res_dict[224].tolist(),
                }
                for seed, res_dict in seeds.items()
            }
            for k_shot, seeds in matched_accs.items()
        },
    }
    tmp_path = PROGRESS_FILE + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(report, f, indent=2)
    os.replace(tmp_path, PROGRESS_FILE)  # rename ذرّي — إما ينجح كامل أو مايحصلش خالص

    # 🆕 طباعة تذكير بعد كل حفظ — عشان تفتكر تعمل commit بانتظام
    print(f"💾 اتحفظ التقدّم محليًا في {PROGRESS_FILE} — "
          f"متنساش تعمل 'Save Version' من وقت للتاني عشان التقدّم يتحفظ فعليًا.")

# ============================================
# 8c. AUTO-PUSH TO GITHUB (بديل تلقائي للتنزيل اليدوي)
# ============================================
# بعد كل seed، بنعمل push لكل محتوى WORK_ROOT (progress.json + كل الـ
# checkpoints) لمستودع GitHub بتاعك. لو فشل لأي سبب، بنطبع تحذير ونكمّل
# التدريب عادي — الفشل هنا مايوقفش التجربة خالص.

_github_push_warned_once = False

# ملحوظة: _run_git_cmd معرّفة فوق في قسم 0 (PATHS) عشان تُستخدم في
# الاسترجاع التلقائي من GitHub قبل بداية التدريب، وبنعيد استخدامها هنا.


def push_results_to_github(commit_message="Update results"):
    """🆕 يعمل push تلقائي لملفات WORK_ROOT (progress.json + checkpoints)
    لمستودع GitHub محدد في GITHUB_REPO. أي فشل هنا بيتحول لتحذير فقط —
    مايوقفش التدريب خالص."""
    global _github_push_warned_once

    if GITHUB_REPO.startswith("your-username/"):
        if not _github_push_warned_once:
            print("ℹ️  GITHUB_REPO لسه مش معدّل (لسه على القيمة الافتراضية) — "
                  "هنتجاهل GitHub Auto-Push. غيّر GITHUB_REPO في بداية الكود "
                  "لو عايز تفعّل الخاصية دي.")
            _github_push_warned_once = True
        return False

    try:
        import subprocess as _sp  # noqa: F401 (تأكيد إنه متاح هنا)
    except ImportError:
        pass

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print(f"⚠️  مقدرتش أجيب GITHUB_TOKEN من Kaggle Secrets ({e}). "
              f"اتأكد إنك ضفته من Add-ons → Secrets باسم GITHUB_TOKEN. "
              f"هنتجاهل GitHub Auto-Push المرة دي.")
        return False

    remote_url = f"https://{token}@github.com/{GITHUB_REPO}.git"

    try:
        # Clone لو أول مرة، وإلا pull عادي
        if not os.path.exists(os.path.join(GITHUB_CLONE_DIR, ".git")):
            print(f"\n📥 بنعمل clone لأول مرة لمستودع النتائج ({GITHUB_REPO})...")
            ok, out = _run_git_cmd([
                "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                remote_url, GITHUB_CLONE_DIR
            ])
            if not ok:
                # ممكن يكون البranch مش موجود لسه (مستودع فاضي) → نعمل init بدالها
                print("ℹ️  الـ clone فشل (غالبًا المستودع فاضي لسه) — هنعمل init محلي بدل كده.")
                os.makedirs(GITHUB_CLONE_DIR, exist_ok=True)
                _run_git_cmd(["init"], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["checkout", "-b", GITHUB_BRANCH], cwd=GITHUB_CLONE_DIR)
                _run_git_cmd(["remote", "add", "origin", remote_url], cwd=GITHUB_CLONE_DIR)
        else:
            ok, out = _run_git_cmd(["pull", "--rebase", "origin", GITHUB_BRANCH],
                                    cwd=GITHUB_CLONE_DIR)
            if not ok:
                print(f"⚠️  git pull فشل (هنكمّل ونحاول الـ push عادي):\n{out}")

        _run_git_cmd(["config", "user.email", "kaggle-bot@example.com"], cwd=GITHUB_CLONE_DIR)
        _run_git_cmd(["config", "user.name", "Kaggle Auto-Push"], cwd=GITHUB_CLONE_DIR)

        # ننسخ كل محتوى WORK_ROOT لمجلد فرعي جوه المستودع
        dst_dir = os.path.join(GITHUB_CLONE_DIR, GITHUB_SUBFOLDER)
        os.makedirs(dst_dir, exist_ok=True)
        for item in os.listdir(WORK_ROOT):
            src_path = os.path.join(WORK_ROOT, item)
            dst_path = os.path.join(dst_dir, item)
            if os.path.isfile(src_path):
                shutil.copy2(src_path, dst_path)

        ok, out = _run_git_cmd(["add", "-A"], cwd=GITHUB_CLONE_DIR)
        ok, out = _run_git_cmd(["commit", "-m", commit_message], cwd=GITHUB_CLONE_DIR)
        if not ok and "nothing to commit" not in out.lower():
            print(f"⚠️  git commit فشل:\n{out}")
            return False

        ok, out = _run_git_cmd(
            ["push", "-u", "origin", f"HEAD:{GITHUB_BRANCH}"],
            cwd=GITHUB_CLONE_DIR,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        )
        if not ok:
            print(f"⚠️  git push فشل (هنكمّل التدريب عادي، الملفات لسه محفوظة "
                  f"محليًا في /kaggle/working):\n{out}")
            return False

        print(f"🚀 اتعمل push ناجح للمستودع {GITHUB_REPO} (فرع {GITHUB_BRANCH}) — "
              f"الملفات دلوقتي متاحة على GitHub.")
        return True

    except Exception as e:
        print(f"⚠️  GitHub Auto-Push فشل بشكل غير متوقع ({e}) — هنكمّل التدريب عادي.")
        return False


# ============================================
# 9. MAIN — LOOP OVER INPUT SIZES × 3 SEEDS
# ============================================

if __name__ == "__main__":
    INPUT_SIZES = [84, 224]
    SEEDS       = [1, 2, 3]
    N_WAY       = 5
    Q_QUERY     = 15
    N_MATCHED_EPISODES = 2000   # عدد الـ episodes الثابتة للمقارنة الـ paired
    MATCHED_EPISODES_SEED = 999  # seed مستقل عن seeds التدريب — نفس الـ episodes تُستخدم مع كل الـ seeds

    all_results, matched_accs = load_progress()  # {input_size: {seed: {k_shot: {...}}}}

    for input_size in INPUT_SIZES:
        all_results.setdefault(input_size, {})
        for seed in SEEDS:
            already_done = (
                seed in all_results[input_size]
                and 1 in all_results[input_size][seed]
                and 5 in all_results[input_size][seed]
            )
            if already_done:
                print(f"✅ {input_size}px seed={seed} موجود في التقدّم المحفوظ — تخطّي.")
                continue

            res = run_experiment(input_size, seed, epochs=STAGE1_EPOCHS, episodes=2000)
            all_results[input_size][seed] = res
            save_progress(all_results, matched_accs)  # حفظ فوري بعد كل seed
            # 🆕 بعد ما الـ seed يخلص بالكامل، نعمل push تلقائي لـ GitHub
            # (بيشمل backbone + best checkpoints + progress.json بتاعة الـ
            # seed ده، عشان تقدر تنزّلهم على جهازك براحتك من GitHub)
            push_results_to_github(
                commit_message=f"Results: {input_size}px seed={seed} done"
            )

    # ── ملخص النتائج عبر الـ 3 seeds ──────────
    print("\n" + "="*60)
    print("📊 FINAL SUMMARY — MEAN ± STD OVER 3 SEEDS")
    print("="*60)
    for input_size in INPUT_SIZES:
        for k_shot in [1, 5]:
            accs = [all_results[input_size][s][k_shot]["mean_acc"] for s in SEEDS]
            print(f"  {input_size}x{input_size} | {k_shot}-shot: "
                  f"{np.mean(accs):.2%} ± {np.std(accs):.2%}  "
                  f"(runs: {[f'{a:.2%}' for a in accs]})")

    # ============================================
    # MATCHED-EPISODE EVALUATION (84x84 vs 224x224)
    # ============================================
    print("\n" + "="*60)
    print("📈 MATCHED-EPISODE EVALUATION (paired 84x84 vs 224x224)")
    print("="*60)

    with StageTimer("Stage 4: Matched-Episode Generation"):
        fixed_episodes = {
            k_shot: generate_fixed_episodes(
                root=os.path.join(CUB_ROOT, "test"),
                n_way=N_WAY, k_shot=k_shot, q_query=Q_QUERY,
                n_episodes=N_MATCHED_EPISODES, seed=MATCHED_EPISODES_SEED
            )
            for k_shot in [1, 5]
        }

    _, test_transform_84  = build_transforms(84)
    _, test_transform_224 = build_transforms(224)

    with StageTimer("Stage 5: Matched-Episode Evaluation"):
        for k_shot in [1, 5]:
            matched_accs.setdefault(k_shot, {})
            eps = fixed_episodes[k_shot]

            for seed in SEEDS:
                if seed in matched_accs[k_shot]:
                    print(f"✅ Matched-Eval {k_shot}-shot seed={seed} "
                          f"موجود في التقدّم المحفوظ — تخطّي.")
                    continue

                ckpt_84  = all_results[84][seed][k_shot]["ckpt_path"]
                ckpt_224 = all_results[224][seed][k_shot]["ckpt_path"]

                model_84  = build_protonet_for_inference(ckpt_84,  num_classes=NUM_TRAIN_CLASSES)
                model_224 = build_protonet_for_inference(ckpt_224, num_classes=NUM_TRAIN_CLASSES)

                acc_84  = evaluate_fixed_episodes(model_84,  eps, test_transform_84,
                                                   N_WAY, k_shot, desc=f"84px seed{seed} {k_shot}shot")
                acc_224 = evaluate_fixed_episodes(model_224, eps, test_transform_224,
                                                   N_WAY, k_shot, desc=f"224px seed{seed} {k_shot}shot")

                matched_accs[k_shot][seed] = {84: acc_84, 224: acc_224}
                save_progress(all_results, matched_accs)  # حفظ فوري بعد كل seed
                # 🆕 push تلقائي لـ GitHub بعد كل matched-eval لسيد كذا
                push_results_to_github(
                    commit_message=f"Matched-eval: {k_shot}-shot seed={seed} done"
                )

                paired_resolution_test(acc_84, acc_224,
                                        label=f"{k_shot}-shot, seed={seed}")

            # ── تجميع كل الـ seeds مع بعض (نفس الـ episodes الثابتة مُعادة الاستخدام
            #    مع كل seed، فالـ pairing يفضل صحيح لكل عنصر) ──
            pooled_84  = np.concatenate([matched_accs[k_shot][s][84]  for s in SEEDS])
            pooled_224 = np.concatenate([matched_accs[k_shot][s][224] for s in SEEDS])
            paired_resolution_test(pooled_84, pooled_224,
                                    label=f"{k_shot}-shot, POOLED across {len(SEEDS)} seeds "
                                          f"({N_MATCHED_EPISODES} episodes × {len(SEEDS)} seeds)")

    print_stage_time_summary()
    print(f"\n💾 التقدّم والنتائج محفوظة محليًا في: {PROGRESS_FILE}")
    print("⚠️  متنساش تعمل 'Save Version' (Save & Run All → Commit) دلوقتي "
          "عشان التقدّم ده يتحفظ فعليًا كـ Output تقدر ترجعله لاحقًا.")